# Calibration Analysis

Post-evaluation calibration deep-dive: reliability diagram, ECE, and abstention threshold tuning.

In [ ]:
import sys, json
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from trial_matcher.evaluation.metrics import (
    compute_ece,
    generate_reliability_diagram_data,
    compute_coverage_accuracy_curve,
)

In [ ]:
# Load eval results
results_path = Path('../results/n2c2_eval_latest.json')
if results_path.exists():
    eval_data = json.loads(results_path.read_text())
    print(f'Macro F1: {eval_data["macro_f1"]:.3f}')
    print(f'ECE: {eval_data["ece"]:.3f}')
    print(f'Coverage: {eval_data["overall_coverage"]:.1%}')
else:
    print('No eval results found. Run scripts/run_n2c2_eval.py first.')

In [ ]:
# Reliability diagram
cal_path = Path('../results/calibration_data.json')
if cal_path.exists():
    cal = json.loads(cal_path.read_text())
    
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
    ax.plot(cal['mean_confidence'], cal['accuracy'], 'o-', color='steelblue', label='Model')
    
    # Size circles by bin count
    counts = cal['counts']
    max_count = max(counts) if counts else 1
    sizes = [50 + 200 * (c / max_count) for c in counts]
    ax.scatter(cal['mean_confidence'], cal['accuracy'], s=sizes, 
               color='steelblue', alpha=0.5, zorder=5)
    
    ax.set_xlabel('Mean Confidence')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Reliability Diagram (ECE = {eval_data["ece"]:.3f})')
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig('../results/calibration_plot.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Coverage-accuracy tradeoff by abstention threshold
cov_path = Path('../results/coverage_accuracy_curve.json')
if cov_path.exists():
    curve = json.loads(cov_path.read_text())
    
    fig, ax1 = plt.subplots(figsize=(8, 5))
    ax2 = ax1.twinx()
    
    ax1.plot(curve['thresholds'], curve['accuracy'], 'g-', label='Accuracy')
    ax2.plot(curve['thresholds'], curve['coverage'], 'b--', label='Coverage')
    
    ax1.axvline(0.35, color='red', linestyle=':', alpha=0.7, label='Default threshold')
    
    ax1.set_xlabel('Abstention Threshold')
    ax1.set_ylabel('Accuracy', color='green')
    ax2.set_ylabel('Coverage', color='blue')
    ax1.set_title('Coverage-Accuracy Tradeoff')
    plt.tight_layout()
    plt.savefig('../results/coverage_accuracy_curve.png', dpi=150, bbox_inches='tight')
    plt.show()